# Graph Analysis: Clustering & Community Detection

This notebook demonstrates associo's graph algorithms:

1. **Affinity Propagation** clustering
2. **Louvain** community detection
3. **Label Propagation** (non-overlapping & overlapping)
4. **Connected Components**
5. **Maximal Cliques** & **K-Clique Communities**

All algorithms take a similarity edge list and return item-to-group assignments.

In [ ]:
import polars as pl
from associo import (
    combinatorial_associations,
    clusters,
    communities,
    label_propagation,
    label_propagation_overlapping,
    connected_components,
    maximal_cliques,
    k_clique_communities,
)

## Build a Similarity Matrix from Transactions

First, compute associations to get a Jaccard similarity between products.

In [ ]:
import random
random.seed(42)

# Products grouped into natural clusters
product_groups = {
    "breakfast": ["cereal", "milk", "yogurt", "granola", "orange_juice", "toast"],
    "italian": ["pasta", "tomato_sauce", "parmesan", "olive_oil", "garlic", "basil"],
    "snacks": ["chips", "salsa", "beer", "pretzels", "popcorn", "soda"],
    "baking": ["flour", "sugar", "eggs", "butter", "vanilla", "baking_powder"],
}

orders = []
for order_id in range(1, 801):
    items = set()
    # Pick 1-2 groups, take 3-5 items from each
    for group in random.sample(list(product_groups.values()), k=random.randint(1, 2)):
        items.update(random.sample(group, k=random.randint(3, min(5, len(group)))))
    for item in items:
        orders.append({"product": item, "order_id": order_id})

df = pl.DataFrame(orders)
print(f"Orders: {df['order_id'].n_unique()}, Products: {df['product'].n_unique()}")

In [ ]:
# Compute associations and extract Jaccard similarity
assoc = combinatorial_associations(df, column_items="product", column_tid="order_id")

similarity = (
    assoc
    .select("lhs", "rhs", "jaccard")
    .filter(pl.col("jaccard").is_not_null() & (pl.col("jaccard") > 0))
    .rename({"jaccard": "sim"})
)

print(f"Similarity edges: {len(similarity)}")
similarity.sort("sim", descending=True).head(10)

## 1. Affinity Propagation Clustering

Automatically determines the number of clusters. `cluster_preference_factor` controls granularity (0-100).

In [ ]:
cl = clusters(
    similarity,
    column_lhs="lhs",
    column_rhs="rhs",
    column_similarity="sim",
    cluster_preference_factor=30,  # lower = fewer, larger clusters
)

print(f"Number of clusters: {cl['cluster_label'].n_unique()}")
cl.sort("cluster_label", "item")

In [ ]:
# View cluster contents
for label in sorted(cl["cluster_label"].unique().to_list()):
    items = cl.filter(pl.col("cluster_label") == label)["item"].to_list()
    print(f"Cluster {label}: {', '.join(sorted(items))}")

## 2. Louvain Community Detection

Resolution parameter controls community size: higher values → more, smaller communities.

In [ ]:
comm = communities(
    similarity,
    column_lhs="lhs",
    column_rhs="rhs",
    column_similarity="sim",
    community_resolution=1.0,
)

print(f"Number of communities: {comm['community_label'].n_unique()}")
for label in sorted(comm["community_label"].unique().to_list()):
    items = comm.filter(pl.col("community_label") == label)["item"].to_list()
    print(f"Community {label}: {', '.join(sorted(items))}")

In [ ]:
# Higher resolution → more granular communities
comm_fine = communities(
    similarity,
    column_lhs="lhs",
    column_rhs="rhs",
    column_similarity="sim",
    community_resolution=2.0,
)

print(f"Communities (resolution=2.0): {comm_fine['community_label'].n_unique()}")
for label in sorted(comm_fine["community_label"].unique().to_list()):
    items = comm_fine.filter(pl.col("community_label") == label)["item"].to_list()
    print(f"  Community {label}: {', '.join(sorted(items))}")

## 3. Label Propagation

Fast community detection suitable for large graphs.

In [ ]:
# Non-overlapping
lp = label_propagation(
    similarity,
    column_lhs="lhs",
    column_rhs="rhs",
    column_similarity="sim",
)

print(f"Label Propagation communities: {lp['community_label'].n_unique()}")
for label in sorted(lp["community_label"].unique().to_list()):
    items = lp.filter(pl.col("community_label") == label)["item"].to_list()
    print(f"  Community {label}: {', '.join(sorted(items))}")

In [ ]:
# Overlapping (SLPA-like): items can belong to multiple communities
lp_overlap = label_propagation_overlapping(
    similarity,
    column_lhs="lhs",
    column_rhs="rhs",
    column_similarity="sim",
    n_iter=20,
    threshold=0.1,
)

# Find items that belong to multiple communities
multi = (
    lp_overlap
    .group_by("item")
    .agg(pl.col("community_label").n_unique().alias("n_communities"))
    .filter(pl.col("n_communities") > 1)
    .sort("n_communities", descending=True)
)

print(f"Items in multiple communities: {len(multi)}")
if len(multi) > 0:
    print(multi)

## 4. Connected Components

Find isolated groups of items that have no connections between them.

In [ ]:
# With a high min_edge_weight, weakly connected items get separated
comp = connected_components(
    similarity,
    column_lhs="lhs",
    column_rhs="rhs",
    column_similarity="sim",
    min_edge_weight=0.3,  # only keep strong edges
)

print(f"Components (min_weight=0.3): {comp['component_label'].n_unique()}")
for label in sorted(comp["component_label"].unique().to_list()):
    items = comp.filter(pl.col("component_label") == label)["item"].to_list()
    print(f"  Component {label}: {', '.join(sorted(items))}")

## 5. Clique-Based Methods

Cliques are fully-connected subgraphs. Items can belong to multiple cliques.

In [ ]:
# Maximal cliques
mc = maximal_cliques(
    similarity,
    column_lhs="lhs",
    column_rhs="rhs",
    column_similarity="sim",
    min_clique_size=3,
    min_edge_weight=0.15,
)

print(f"Maximal cliques found: {mc['clique_label'].n_unique()}")
for label in sorted(mc["clique_label"].unique().to_list()):
    items = mc.filter(pl.col("clique_label") == label)["item"].to_list()
    print(f"  Clique {label} ({len(items)} items): {', '.join(sorted(items))}")

In [ ]:
# K-clique communities (clique percolation)
kc = k_clique_communities(
    similarity,
    column_lhs="lhs",
    column_rhs="rhs",
    column_similarity="sim",
    k=3,
    min_edge_weight=0.15,
)

print(f"K-clique communities: {kc['community_label'].n_unique() if len(kc) > 0 else 0}")
if len(kc) > 0:
    for label in sorted(kc["community_label"].unique().to_list()):
        items = kc.filter(pl.col("community_label") == label)["item"].to_list()
        print(f"  Community {label}: {', '.join(sorted(items))}")

## 6. Comparing Algorithms

Each algorithm has different properties — let's compare their outputs side by side.

In [ ]:
# Combine results for comparison
comparison = (
    cl.rename({"cluster_label": "affinity_prop"})
    .join(
        comm.rename({"community_label": "louvain"}),
        on="item",
        how="outer",
        coalesce=True,
    )
    .join(
        lp.rename({"community_label": "label_prop"}),
        on="item",
        how="outer",
        coalesce=True,
    )
    .sort("item")
)

comparison